# TV3: Bước 3 - Information Extraction (IE) – Fact Extraction
## Module 8: Trích xuất Thực thể, Số liệu và Mốc thời gian

---

### Mục tiêu:
BM25 chỉ đo lường sự trùng lặp từ khóa, không phân biệt được sự kiện đúng hay sai về mặt thực tế.
Module 8 trích xuất 3 nhóm đặc trưng Fact quan trọng:
1. **Named Entities:** Người, Tổ chức, Địa danh, Sự kiện bằng `VnCoreNLP NER`.
2. **Temporal Facts:** Ngày, tháng, năm, thế kỷ, quý bằng Regex.
3. **Numerical Facts:** Tỷ lệ %, số lượng, đơn vị đo lường bằng Regex (kèm kỹ thuật Temporal Masking).
Từ đó tính toán:
$$\text{Fact Score} = 0.4 \times \text{Entity Match} + 0.4 \times \text{Number Match} + 0.2 \times \text{Date Match}$$
Xuất bảng kết quả vào: `outputs/evidence_features.csv`.

In [1]:
import os
import re
import sys
import time
from pathlib import Path
import pandas as pd
import py_vncorenlp

PROJECT_ROOT = Path("../..").resolve()
INPUT_PATH = OUTPUT_DIR / "bm25_results.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/retrieval"

# Thiết lập JVM
conda_prefix = Path(sys.prefix)
os.environ["JAVA_HOME"] = str(conda_prefix)
jvm_path = conda_prefix / "lib" / "server" / "libjvm.dylib"
if jvm_path.exists():
    os.environ["JVM_PATH"] = str(jvm_path)

orig_cwd = Path.cwd()
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg", "pos", "ner"], save_dir=str(Path.home() / ".vncorenlp"))
os.chdir(orig_cwd)
print("✓ Khởi tạo VnCoreNLP NER thành công và bảo vệ working directory an toàn!")

2026-09-07 00:22:41 INFO  WordSegmenter:24 - Loading Word Segmentation model
2026-09-07 00:22:41 INFO  PosTagger:23 - Loading POS Tagging model
2026-09-07 00:22:44 INFO  NerRecognizer:34 - Loading NER model
✓ Khởi tạo VnCoreNLP NER thành công và bảo vệ working directory an toàn!


In [2]:
TEMPORAL_PATTERNS = [
    r"\b(?:ngày\s+)?\d{1,2}[\./\-]\d{1,2}[\./\-]\d{4}\b",
    r"\b\d{1,2}\s+tháng\s+\d{1,2}(?:\s+năm\s+\d{4})?\b",
    r"\btháng\s+\d{1,2}[\./\-]\d{4}\b",
    r"\btháng\s+\d{1,2}\s+năm\s+\d{4}\b",
    r"\bquý\s+[1-4](?:[\s/]+năm\s+\d{4}|\s+năm\s+\d{4}|\s+\d{4})?\b",
    r"\bthế\s+kỷ\s+(?:[ivxldcm]+|\d+)\b",
    r"\b(?:năm\s+)?(18\d{2}|19\d{2}|20\d{2})\b",
]

def extract_temporal_facts(text: str) -> set[str]:
    if not isinstance(text, str):
        return set()
    dates = set()
    text_l = text.lower()
    for pat in TEMPORAL_PATTERNS:
        for m in re.finditer(pat, text_l):
            dates.add(m.group().strip())
    return dates

def extract_numerical_facts(text: str) -> set[str]:
    if not isinstance(text, str):
        return set()
    text_l = text.lower()
    for pat in TEMPORAL_PATTERNS:
        text_l = re.sub(pat, " ", text_l)
    nums = set()
    for m in re.finditer(r"\b\d+(?:[\.,]\d+)?\s*%", text_l):
        nums.add(m.group().replace(" ", ""))
    text_l = re.sub(r"\b\d+(?:[\.,]\d+)?\s*%", " ", text_l)
    units = r"(?:triệu|tỷ|nghìn|ngàn|vạn|đô|usd|eur|đồng|vnd|km|m2|m3|m|cm|kg|tấn|lít|héc-ta|ha|người|ca|bệnh nhân|bàn|điểm|tuổi|năm tù)"
    for m in re.finditer(rf"\b\d+(?:[\.,]\d+)?\s*{units}\b", text_l):
        nums.add(m.group().strip())
    text_l = re.sub(rf"\b\d+(?:[\.,]\d+)?\s*{units}\b", " ", text_l)
    for m in re.finditer(r"\b\d+(?:[\.,]\d+)?\b", text_l):
        nums.add(m.group().strip())
    return nums

def extract_entities_from_ann(annotated_dict: dict) -> set[str]:
    ents = set()
    cur = []
    for s_idx, tokens in annotated_dict.items():
        for tok in tokens:
            ner = tok["nerLabel"]
            word = tok["wordForm"].replace("_", " ").lower()
            if ner.startswith("B-"):
                if cur:
                    ents.add(" ".join(cur))
                cur = [word]
            elif ner.startswith("I-") and cur:
                cur.append(word)
            else:
                if cur:
                    ents.add(" ".join(cur))
                    cur = []
        if cur:
            ents.add(" ".join(cur))
            cur = []
    return ents

print("✓ Đã định nghĩa các hàm trích xuất Fact.")

✓ Đã định nghĩa các hàm trích xuất Fact.


In [3]:
df_bm25 = pd.read_csv(INPUT_PATH)
unique_sents = list(set(df_bm25["claim"].unique()).union(set(df_bm25["retrieved_evidence"].unique())))
print(f"• Trích xuất đặc trưng cho {len(unique_sents):,} câu phân biệt...")

feature_cache = {}
t0 = time.time()
for s in unique_sents:
    ann = rdrsegmenter.annotate_text(s)
    feature_cache[s] = {
        "entities": extract_entities_from_ann(ann),
        "dates": extract_temporal_facts(s),
        "numbers": extract_numerical_facts(s)
    }
os.chdir(orig_cwd)
print(f"✓ Hoàn tất trích xuất đặc trưng trong {time.time() - t0:.2f}s!")

def match_score(c_set, e_set, ev_text):
    if not c_set:
        return 0.5
    ev_l = ev_text.lower()
    hits = sum(1 for x in c_set if x in e_set or x in ev_l)
    return hits / len(c_set)

ent_m, num_m, date_m, fact_s = [], [], [], []
for _, row in df_bm25.iterrows():
    c = feature_cache[row["claim"]]
    e = feature_cache[row["retrieved_evidence"]]
    ev = row["retrieved_evidence"]
    em = match_score(c["entities"], e["entities"], ev)
    nm = match_score(c["numbers"], e["numbers"], ev)
    dm = match_score(c["dates"], e["dates"], ev)
    fs = 0.4 * em + 0.4 * nm + 0.2 * dm
    ent_m.append(round(em, 4))
    num_m.append(round(nm, 4))
    date_m.append(round(dm, 4))
    fact_s.append(round(fs, 4))

df_bm25["entity_match"] = ent_m
df_bm25["number_match"] = num_m
df_bm25["date_match"] = date_m
df_bm25["fact_score"] = fact_s

out_feat_path = OUTPUT_DIR / "evidence_features.csv"
df_bm25.to_csv(out_feat_path, index=False)
print(f"✓ Đã lưu file: {out_feat_path.name}")
display(df_bm25[["claim", "retrieved_evidence", "entity_match", "number_match", "date_match", "fact_score"]].head(5))

• Trích xuất đặc trưng cho 3,757 câu phân biệt...


✓ Hoàn tất trích xuất đặc trưng trong 7.45s!
✓ Đã lưu file: evidence_features.csv


,claim,retrieved_evidence,entity_match,number_match,date_match,fact_score
0,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...,1.0000,0.5,0.0,0.6000
1,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Trong số đó, có vua hề Charlie Chaplin (vua hề...",0.3333,0.5,0.0,0.3333
2,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Swore Oath ở khách sạn Saigon Morin năm 2004 T...,0.3333,0.5,0.0,0.3333
3,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Phu nhân cựu Tổng thống Pháp, bà Bernadette Ch...",0.0000,0.5,0.0,0.2000
4,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Saigon Morin, khách sạn 4 sao hàng đầu tại Huế...",0.0000,0.5,0.0,0.2000
